# diag0 — BiMamba 역방향 브랜치 상수성 확인

**묻는 것:** BiMamba 의 역방향(backward) 스택이 관측 정보를 실제로 나르는가?

**코드 구조상의 예측:**

- `decoder_in = torch.zeros(...)` 이고 `decoder_pos_embed` 는 `nn.Embedding(K, D)` → 쿼리 `Q` 는 **배치 무관 상수**
- 디코더 입력은 `[C ; Q]` 이고 `C` 만 관측 의존
- 역방향 브랜치는 `flip([C ; Q]) = [q_K…q_1, c_M…c_1]` 을 스캔 → **쿼리가 시퀀스 맨 앞**
- Mamba-2 스캔은 causal (`_causal_conv_step` + `mamba_chunk_scan_combined`)

→ 역방향 스택의 쿼리 위치 출력은 `Q` 만의 함수, 즉 **관측·태스크와 무관한 상수**여야 한다.

| 측정 | 기대 | 뜻 |
|---|---|---|
| `BWD @ query` | `0` | 역방향 출력이 상수 |
| `FWD @ query` | `>> 0` | 민감도 대조군 |
| `출력 action` | `>> 0` | 모델이 살아있음 |

`FWD` 까지 0 이면 두 배치가 사실 같은 관측이라는 뜻이라 **판정 불가**로 빠진다 (거짓 확정 방지).

덤으로 `||bwd|| / ||fwd||` 를 위치별로 잰다. 역방향 기여가 크기 자체로 무시할 만한지 보기 위한 것 —
이 값이 유의미하면 "긴 스캔이 위치 정체성을 잃고 BiMamba 가 head 직전에 복원한다" 가설로 간다.


## 0) 부팅

In [ ]:
import importlib, os, sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents)
          if (c / 'notebooks' / 'libero' / 'diag0_backward_const.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
try:
    import lerobot  # noqa: F401
except ImportError:
    sys.path.insert(0, str(_r / 'src'))

import torch
import diag0_backward_const as D
D = importlib.reload(D)

print('repo :', _r)
print('cuda :', torch.cuda.is_available(), torch.cuda.device_count(), '장')
print('tags :', D.K_TAGS)

## 1) 설정

`bimamba_pure` 가 논문의 BiMamba 다. **`bimamba` 는 carry 붙은 옛 bimos 이므로 쓰지 않는다.**

In [ ]:
SEED   = 0
STEP   = 150_000      # None 이면 최신 체크포인트
TASK   = 'libero_10'
BATCH  = 4            # 관측이 서로 다른 샘플 수
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SHARE = Path(os.environ.get('LEROBOT_OUTPUT', Path.home() / 'lerobot_project' / 'outputs'))
SHARE = SHARE / 'final' / 'share' / 'diag0'
SHARE.mkdir(parents=True, exist_ok=True)
print(SHARE)

## 2) K=100 단일 확인 — 이 셀 하나가 0단계의 답이다

In [ ]:
res100 = D.run(tag='bimamba_pure', seed=SEED, step=STEP, task=TASK,
               batch=BATCH, device=DEVICE)

## 3) K 별로 — 50 / 100 / 150

체크포인트가 없는 K 는 건너뛴다. 역방향 기여 비율이 K 에 따라 커지면,
그 자체로 성공률 이득의 K 의존성(+2.2 → +8.2 → +9.0)을 설명하는 근거가 된다.

In [ ]:
results = {}
for K, tag in D.K_TAGS.items():
    print('\n' + '#' * 68 + f'\n# {tag}  (K={K})\n' + '#' * 68)
    try:
        results[K] = D.run(tag=tag, seed=SEED, step=STEP, task=TASK,
                           batch=BATCH, device=DEVICE)
    except FileNotFoundError as e:
        print(f'건너뜀 — {e}')

## 4) 요약표

In [ ]:
hdr = f"{'K':>5} {'tag':<20} {'판정':<14} {'BWD A-B':>11} {'FWD A-B':>11} {'ratio 평균':>11}"
print(hdr); print('-' * len(hdr))
rows = []
for K in sorted(results):
    r = results[K]
    rat = r['ratio'].mean().item() if 'ratio' in r else float('nan')
    print(f"{K:>5} {r['tag']:<20} {r['verdict']:<14} "
          f"{r.get('bwd_ab', float('nan')):11.3e} {r.get('fwd_ab', float('nan')):11.3e} {rat:11.3f}")
    rows.append({'K': K, 'tag': r['tag'], 'verdict': r['verdict'],
                 'bwd_ab': r.get('bwd_ab'), 'bwd_spread': r.get('bwd_spread'),
                 'fwd_ab': r.get('fwd_ab'), 'fwd_spread': r.get('fwd_spread'),
                 'act_ab': r.get('act_ab'), 'ratio_mean': rat})

import csv, time
STAMP = time.strftime('%Y%m%d_%H%M')
csv_path = SHARE / f'diag0_summary_{STAMP}.csv'
if rows:
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    print('\nsaved', csv_path)

## 5) 위치별 역방향 기여 — `||bwd|| / ||fwd||`

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for K in sorted(results):
    r = results[K]
    if 'ratio' not in r:
        continue
    y = r['ratio'].numpy()
    ax.plot(range(1, len(y) + 1), y, label=f"K={K}", lw=1.6)
ax.set_xlabel('청크 내 위치 k'); ax.set_ylabel('||bwd|| / ||fwd||')
ax.set_title('역방향 브랜치 기여 크기 (위치별)')
ax.grid(alpha=.3); ax.legend()
fig.tight_layout()
png = SHARE / f'diag0_ratio_{STAMP}.png'
fig.savefig(png, dpi=150, bbox_inches='tight')
print('saved', png)
plt.show()

## 6) 결과 읽는 법

**`BWD A-B == 0` 이고 `배치 내 샘플 간 == 0`** → 확정. 역방향 스택은 정보를 나르지 않고
head 직전에 위치별 상수벡터 `b_k` 를 더하는 역할만 한다. 원고의 "앞쪽 쿼리가 뒤쪽을 본다"
설명은 이 구현에서 성립하지 않으며, 1단계 위치별 오차 분석의 해석 틀이 여기서 정해진다.

**`ratio` 가 0.01 수준** → 역방향 기여 자체가 미미하다는 뜻. 위치 정체성 가설이 약해지고,
9.0 포인트 차이의 원인을 다른 데서 찾아야 한다.

**`ratio` 가 0.3~1 수준이고 K 에 따라 커짐** → 가설이 강해진다. 다음은 2단계 ablation —
역방향 스택을 `nn.Embedding(K, 512)` 하나로 대체해서 이득이 회복되는지 본다.

**`반증`** → 구조 분석이 틀렸다. 스캔이 causal 이 아니거나 쿼리가 관측 의존이라는 뜻이므로
`modeling_acm2_sscp_literal.py` 의 `mamba2_stateful_forward` 경로를 다시 봐야 한다.

결과(위 표 + PNG + CSV)는 `outputs/final/share/diag0/` 에 남는다.